In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [8]:
!pip install google-genai -q

from kaggle_secrets import UserSecretsClient
import os
os.environ["GOOGLE_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_API_KEY")

from google import genai
client = genai.Client()
resp = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Say hello in one word."
)
print(resp.text)

Hi


In [3]:
!pip install google-adk -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.0/306.0 kB 1.9 MB/s eta 0:00:00a 0:00:01


In [9]:
!pip install pytest -q

import subprocess
from google.adk.tools.tool_context import ToolContext

CODE_FILE = "solution.py"
TEST_FILE = "test_solution.py"

def run_pytest(tool_context: ToolContext) -> dict:
    """Write the code and tests to files, then run pytest. Returns pass/fail."""
    code = tool_context.state.get("generated_code", "").replace("```python", "").replace("```", "").strip()
    tests = tool_context.state.get("test_code", "").replace("```python", "").replace("```", "").strip()

    with open(CODE_FILE, "w") as f:
        f.write(code)
    with open(TEST_FILE, "w") as f:
        f.write(tests)

    result = subprocess.run(["pytest", TEST_FILE, "-q"], capture_output=True, text=True)
    passed = result.returncode == 0
    output = (result.stdout or "") + (result.stderr or "")
    tool_context.state["test_results"] = output
    return {"tests_passed": passed, "output": output}

def exit_loop(tool_context: ToolContext) -> dict:
    """Stop the loop. Call this only when tests pass."""
    tool_context.actions.escalate = True
    tool_context.actions.skip_summarization = True
    return {}

print("Tools ready.")

Tools ready.


In [10]:
!pip install bandit -q

import re
import subprocess
from google.adk.tools.tool_context import ToolContext

def secrets_scan(tool_context: ToolContext) -> dict:
    """Look for hardcoded secrets in the generated code."""
    code = tool_context.state.get("generated_code", "")
    patterns = {
        "OpenAI-style key": r"sk-[A-Za-z0-9]{10,}",
        "AWS access key": r"AKIA[0-9A-Z]{16}",
        "hardcoded password": r"(?i)password\s*=\s*['\"][^'\"]+['\"]",
        "hardcoded api_key": r"(?i)api[_-]?key\s*=\s*['\"][^'\"]+['\"]",
    }
    found = [label for label, p in patterns.items() if re.search(p, code)]
    return {"secrets_found": bool(found), "details": found}

def bandit_scan(tool_context: ToolContext) -> dict:
    """Run bandit (a real Python security linter) on the generated code."""
    code = tool_context.state.get("generated_code", "").replace("```python", "").replace("```", "").strip()
    with open("solution.py", "w") as f:
        f.write(code)
    result = subprocess.run(["bandit", "-r", "solution.py", "-f", "txt"], capture_output=True, text=True)
    return {"bandit_output": result.stdout or "No issues found."}

print("Security tools ready.")

Security tools ready.


In [11]:
from google.adk.agents import LlmAgent, SequentialAgent, LoopAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

MODEL = "gemini-2.5-flash-lite"

planner = LlmAgent(
    name="Planner", model=MODEL, include_contents="none",
    instruction="""You are a software architect for a non-technical founder.
Turn their request into a PRECISE, unambiguous spec. Vague specs cause the
coder and test-writer to disagree, so leave no room for interpretation.
Your spec MUST include:
1. The exact function name and what it takes as input.
2. The EXACT rules, numbered, with no ambiguity. State exact thresholds.
3. The EXACT return format (types and structure), described precisely.
4. 3-5 specific example inputs and their exact expected outputs, so the
   coder and test-writer cannot disagree on behaviour.
Output only the spec.""",
    output_key="spec",
)

coder = LlmAgent(
    name="Coder", model=MODEL, include_contents="none",
    instruction="""You are a Python developer. Write code for this spec:
{spec}
Output ONLY the Python code in a ```python block. No explanation.""",
    output_key="generated_code",
)

test_writer = LlmAgent(
    name="TestWriter", model=MODEL, include_contents="none",
    instruction="""You are a test engineer. Write pytest tests based ONLY on
this spec. Do not invent rules the spec does not state.
{spec}
Rules:
- Write one test for each example input/output pair the spec gives.
- You may add edge cases (empty input, very short input) ONLY using the
  exact rules in the spec. Count failures using the spec's rules precisely.
- If unsure of the expected output, do NOT guess — only test what the spec defines.
- Import with: from solution import <name>
- Output ONLY the Python test code in a ```python block.""",
    output_key="test_code",
)

runner_agent = LlmAgent(
    name="Runner", model=MODEL, include_contents="none",
    instruction="""You run the tests and decide what happens next.
1. Call run_pytest.
2. If tests_passed is True: call exit_loop, then say "All tests passed."
3. If tests_passed is False: say "Tests failed:" and show the failure details.
   Do NOT call exit_loop.""",
    tools=[run_pytest, exit_loop],
    output_key="run_report",
)

coder_fixer = LlmAgent(
    name="CoderFixer", model=MODEL, include_contents="none",
    instruction="""Fix the code based on the test failures.
Current code:
{generated_code}
Test failures to fix:
{test_results}
Fix ONLY what caused the failures. No hardcoded secrets.
Output ONLY the corrected Python code in a ```python block.""",
    output_key="generated_code",
)

correction_loop = LoopAgent(
    name="CorrectionLoop",
    sub_agents=[runner_agent, coder_fixer],
    max_iterations=1, 
    # CHANGE THIS TO 3 FOR DEMO
)

reviewer = LlmAgent(
    name="Reviewer", model=MODEL, include_contents="none",
    instruction="""You are a senior engineer writing a plain-English review
for a non-technical founder who will NOT read the code.

First call secrets_scan, then call bandit_scan.

Then write a short report in this exact format:

Description: One paragraph on what this code does.

Issues:
- Critical: (must-fix, e.g. secrets found)
- Warnings: (smaller risks)
- Best Practices: (suggestions)
- Quick Win: (single most useful improvement, one sentence)

If no issues at all, write "LGTM" after Description.
Use the scan results to decide what is Critical.""",
    tools=[secrets_scan, bandit_scan],
    output_key="review_report",
)

# Rebuild pipeline with Reviewer at the end
root_agent = SequentialAgent(
    name="FounderForge",
    sub_agents=[planner, coder, test_writer, correction_loop, reviewer],
)
print("Pipeline rebuilt with Reviewer.")

async def run(prompt):
    s = InMemorySessionService()
    await s.create_session(app_name="ff", user_id="u1", session_id="s1")
    runner = Runner(agent=root_agent, app_name="ff", session_service=s)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
        if event.is_final_response() and event.content:
            text = event.content.parts[0].text
            # Skip the stray "no fixes needed" message on a passing run
            if event.author == "CoderFixer" and "No fixes needed" in text:
                continue
            print(f"\n--- {event.author} ---")
            print(text)

Pipeline rebuilt with Reviewer.


In [12]:
await run("A function that checks if a password is strong enough")


--- Planner ---
```json
{
  "function_name": "is_strong_password",
  "description": "Checks if a given password meets the strength criteria.",
  "parameters": [
    {
      "name": "password",
      "type": "string",
      "description": "The password string to evaluate."
    }
  ],
  "rules": [
    {
      "order": 1,
      "description": "The password must be at least 10 characters long."
    },
    {
      "order": 2,
      "description": "The password must contain at least one uppercase letter (A-Z)."
    },
    {
      "order": 3,
      "description": "The password must contain at least one lowercase letter (a-z)."
    },
    {
      "order": 4,
      "description": "The password must contain at least one digit (0-9)."
    },
    {
      "order": 5,
      "description": "The password must contain at least one special character from the following set: !@#$%^&*()-=_+|[]{}<>,.?/~"
    }
  ],
  "return_format": {
    "type": "boolean",
    "description": "Returns true if all strength

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [13]:
DEMO RUN — captured July 4


--- Planner ---
```json
{
  "function_name": "is_strong_password",
  "description": "Checks if a given password meets the strength criteria.",
  "parameters": [
    {
      "name": "password",
      "type": "string",
      "description": "The password string to evaluate."
    }
  ],
  "rules": [
    {
      "order": 1,
      "description": "The password must be at least 10 characters long."
    },
    {
      "order": 2,
      "description": "The password must contain at least one uppercase letter (A-Z)."
    },
    {
      "order": 3,
      "description": "The password must contain at least one lowercase letter (a-z)."
    },
    {
      "order": 4,
      "description": "The password must contain at least one digit (0-9)."
    },
    {
      "order": 5,
      "description": "The password must contain at least one special character from the following set: !@#$%^&*()-=_+|[]{}<>,.?/~"
    }
  ],
  "return_format": {
    "type": "boolean",
    "description": "Returns true if all strength criteria are met, false otherwise."
  },
  "examples": [
    {
      "input": {
        "password": "Password123!"
      },
      "output": true
    },
    {
      "input": {
        "password": "password123!"
      },
      "output": false
    },
    {
      "input": {
        "password": "Password!"
      },
      "output": false
    },
    {
      "input": {
        "password": "Pass123!"
      },
      "output": false
    },
    {
      "input": {
        "password": "Password1234567890"
      },
      "output": false
    },
    {
      "input": {
        "password": "P@$$wOrd123"
      },
      "output": true
    }
  ]
}
```

--- Coder ---
```python
import re

def is_strong_password(password: str) -> bool:
  """
  Checks if a given password meets the strength criteria.

  Args:
    password: The password string to evaluate.

  Returns:
    True if all strength criteria are met, false otherwise.
  """

  # Rule 1: The password must be at least 10 characters long.
  if len(password) < 10:
    return False

  # Rule 2: The password must contain at least one uppercase letter (A-Z).
  if not re.search(r"[A-Z]", password):
    return False

  # Rule 3: The password must contain at least one lowercase letter (a-z).
  if not re.search(r"[a-z]", password):
    return False

  # Rule 4: The password must contain at least one digit (0-9).
  if not re.search(r"[0-9]", password):
    return False

  # Rule 5: The password must contain at least one special character.
  if not re.search(r"[!@#$%^&*()\-=_+|\[\]{}<>,.?/~]", password):
    return False

  # If all rules are met, the password is strong.
  return True
```

--- TestWriter ---
```python
import pytest
from solution import is_strong_password

@pytest.mark.parametrize("password, expected", [
    ("Password123!", True),
    ("password123!", False),
    ("Password!", False),
    ("Pass123!", False),
    ("Password1234567890", False),
    ("P@$$wOrd123", True),
])
def test_is_strong_password_examples(password, expected):
    """
    Tests the is_strong_password function with the provided examples.
    """
    assert is_strong_password(password) == expected

def test_is_strong_password_empty():
    """
    Tests the is_strong_password function with an empty string.
    Rule 1: The password must be at least 10 characters long. (Fails)
    """
    assert is_strong_password("") == False

def test_is_strong_password_too_short_no_special_chars():
    """
    Tests a password that is too short but contains all other required character types.
    Rule 1: The password must be at least 10 characters long. (Fails)
    """
    assert is_strong_password("Abc1") == False

def test_is_strong_password_no_uppercase():
    """
    Tests a password that meets length and has other characters but no uppercase.
    Rule 2: The password must contain at least one uppercase letter (A-Z). (Fails)
    """
    assert is_strong_password("password123!@#") == False

def test_is_strong_password_no_lowercase():
    """
    Tests a password that meets length and has other characters but no lowercase.
    Rule 3: The password must contain at least one lowercase letter (a-z). (Fails)
    """
    assert is_strong_password("PASSWORD123!@#") == False

def test_is_strong_password_no_digit():
    """
    Tests a password that meets length and has other characters but no digit.
    Rule 4: The password must contain at least one digit (0-9). (Fails)
    """
    assert is_strong_password("Password!@#abc") == False

def test_is_strong_password_no_special_char():
    """
    Tests a password that meets length and has other characters but no special character.
    Rule 5: The password must contain at least one special character. (Fails)
    """
    assert is_strong_password("Password123abc") == False

def test_is_strong_password_exactly_10_chars_all_types():
    """
    Tests a password that is exactly 10 characters long and meets all other criteria.
    """
    assert is_strong_password("Abcdef123!") == True

def test_is_strong_password_long_with_all_types():
    """
    Tests a longer password that meets all criteria.
    """
    assert is_strong_password("ThisIsAVeryStrongPassword123!@#") == True

def test_is_strong_password_special_chars_from_set():
    """
    Tests a password with a special character that is explicitly in the allowed set.
    """
    assert is_strong_password("Password123.") == True

def test_is_strong_password_special_chars_not_from_set():
    """
    Tests a password with a special character NOT in the allowed set.
    Rule 5: The password must contain at least one special character from the following set: !@#$%^&*()-=_+|[]{}<>,.?/~ (Fails)
    """
    assert is_strong_password("Password123`") == False
```

--- Runner ---
All tests passed.

SyntaxError: invalid syntax (479577032.py, line 1)

In [ ]:
=== Reviewer Report (generated from scanner output) ===

Description: This code implements a password strength checker. It validates
that a password is at least 10 characters long and contains at least one
uppercase letter, one lowercase letter, one digit, and one special character
from a defined set. It returns True if all rules pass, False otherwise.

Issues:
- Critical: None. Secrets scan found no hardcoded credentials. Bandit found
  no security issues at any severity level.
- Warnings: None.
- Best Practices: The regex character class for special characters is
  hard-coded inside the function. For maintainability, consider extracting
  it as a module-level constant so the allowed set can be updated in one place.
- Quick Win: LGTM — code is safe to ship as a self-contained utility. Consider
  adding a small docstring example so downstream users see expected inputs
  at a glance.

Verdict: LGTM.

In [14]:
# Import your tool functions (already defined in an earlier cell)
from google.adk.tools.tool_context import ToolContext

# Fake a minimal tool_context that has state
class FakeContext:
    def __init__(self):
        # Read the generated code from disk
        with open("solution.py", "r") as f:
            self.state = {"generated_code": f.read()}

fake_ctx = FakeContext()

print("=== Secrets Scan ===")
print(secrets_scan(fake_ctx))
print("\n=== Bandit Scan ===")
print(bandit_scan(fake_ctx))

=== Secrets Scan ===
{'secrets_found': False, 'details': []}

=== Bandit Scan ===
{'bandit_output': 'Run started:2026-07-04 23:53:16.750077+00:00\n\nTest results:\n\tNo issues identified.\n\nCode scanned:\n\tTotal lines of code: 20\n\tTotal lines skipped (#nosec): 0\n\tTotal potential issues skipped due to specifically being disabled (e.g., #nosec BXXX): 0\n\nRun metrics:\n\tTotal issues (by severity):\n\t\tUndefined: 0\n\t\tLow: 0\n\t\tMedium: 0\n\t\tHigh: 0\n\tTotal issues (by confidence):\n\t\tUndefined: 0\n\t\tLow: 0\n\t\tMedium: 0\n\t\tHigh: 0\nFiles skipped (0):\n'}
